# 03 · BioLingual: embeddings, zero-shot (M3), linear probe (M4)

**Run only after `PREREGISTRATION.md` is committed.** GPU + Internet on.

Inputs: `birdclef-2025`, the aux dataset (`folds.csv`, `species.csv`), and the output of `02` (`chunks.csv` only — the mel cache is not used here).

Uses exactly the chunks chosen in 02. Each chunk is resampled to 48 kHz and passed to the CLAP feature extractor unpadded (chunks are ≤ 10 s, so it repeat-pads and never randomly crops → deterministic).

Outputs (`/kaggle/working`):
- `emb.npy` — L2-normalised audio embedding per chunk (row order = `chunks.csv`)
- `recordings.csv` — recording order used in all prediction arrays, with `label_idx`, `fold`, `leak`
- `classes.json` — class order (sorted `primary_label`)
- `probs_m3_P1.npy`, `probs_m3_P2.npy`, `probs_m3_P3.npy`, `probs_m4.npy` — `(n_recordings, n_classes)` recording-level probabilities (M4 out-of-fold)

In [ ]:
import json, os
from pathlib import Path

import numpy as np
import pandas as pd

ON_KAGGLE = Path('/kaggle/input').exists()
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INPUT = Path('/kaggle/input') if ON_KAGGLE else REPO

def find_dir(marker):
    hits = sorted(INPUT.rglob(marker))
    assert hits, f'{marker} not found under {INPUT}'
    return hits[0].parent

COMP = find_dir('taxonomy.csv')
AUX = find_dir('species.csv')
FEAT = find_dir('chunks.csv')
OUT = Path('/kaggle/working') if ON_KAGGLE else REPO / 'cache' / 'biolingual'
OUT.mkdir(parents=True, exist_ok=True)

import torch, torchaudio, soundfile as sf
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = 'davidrrobinson/BioLingual'
SR_IN, SR_MODEL = 32000, 48000

species = pd.read_csv(AUX / 'species.csv', index_col=0)
folds = pd.read_csv(AUX / 'folds.csv')
chunks = pd.read_csv(FEAT / 'chunks.csv')
classes = sorted(species.index)
cls_idx = {c: i for i, c in enumerate(classes)}

rec = folds[folds['filename'].isin(chunks['filename'])].reset_index(drop=True)
rec['label_idx'] = rec['primary_label'].map(cls_idx)
assert rec['label_idx'].notna().all()
rec_idx = {f: i for i, f in enumerate(rec['filename'])}
chunks['rec'] = chunks['filename'].map(rec_idx)
print(DEVICE, len(classes), 'classes |', len(rec), 'recordings |', len(chunks), 'chunks')

In [ ]:
from transformers import ClapModel, ClapProcessor

model = ClapModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
processor = ClapProcessor.from_pretrained(MODEL_ID)
fe, tok = processor.feature_extractor, processor.tokenizer
assert fe.sampling_rate == SR_MODEL, fe.sampling_rate
D = model.config.projection_dim

# get_*_features returns a tensor in transformers v4 and an output object in v5.
def as_embed(out, projection):
    if not torch.is_tensor(out):
        out = out.pooler_output
        if out.shape[-1] != D:
            out = projection(out)
    assert out.shape[-1] == D, out.shape
    return out


## 1. Audio embeddings per chunk

In [ ]:
from torch.utils.data import Dataset, DataLoader

by_file = chunks.groupby('filename', sort=False)

class Chunks(Dataset):
    def __init__(self):
        self.files = list(by_file.groups)
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        fn = self.files[i]
        g = by_file.get_group(fn)
        y, sr = sf.read(str(COMP / 'train_audio' / fn), dtype='float32', always_2d=True)
        y = torch.from_numpy(y.mean(axis=1))
        if sr != SR_IN:
            y = torchaudio.functional.resample(y, sr, SR_IN)
        clips = []
        for s in g['start_s']:
            a = int(s * SR_IN)
            clip = y[a:a + 5 * SR_IN]
            clip = torchaudio.functional.resample(clip, SR_IN, SR_MODEL)
            clips.append(clip.numpy())
        feats = fe(clips, sampling_rate=SR_MODEL, return_tensors='pt')
        return g['row'].to_numpy(), feats

def collate(batch):
    rows = np.concatenate([b[0] for b in batch])
    keys = batch[0][1].keys()
    return rows, {k: torch.cat([b[1][k] for b in batch]) for k in keys}

emb = np.zeros((len(chunks), D), dtype=np.float32)
loader = DataLoader(Chunks(), batch_size=16, num_workers=os.cpu_count(), collate_fn=collate)

from tqdm.auto import tqdm
with torch.no_grad():
    for rows, feats in tqdm(loader):
        e = as_embed(model.get_audio_features(**{k: v.to(DEVICE) for k, v in feats.items()}), model.audio_projection)
        emb[rows] = torch.nn.functional.normalize(e, dim=-1).float().cpu().numpy()

assert np.abs(np.linalg.norm(emb, axis=1) - 1).max() < 1e-3, 'some chunks were not embedded'
np.save(OUT / 'emb.npy', emb)
emb.shape

## 2. M3 — zero-shot

Chunk probabilities = softmax over classes of `logit_scale_a · cos(audio, text)` (the model's own audio-side temperature); recording probability = mean over its chunks.

In [ ]:
PROMPTS = {
    'P1': lambda r: f"the sound of a {r.common_name}",
    'P2': lambda r: f"the sound of {r.scientific_name}",
    'P3': lambda r: f"the sound of a {r.common_name}, {r.scientific_name}",
}
scale = model.logit_scale_a.exp().item()
chunk_rec = chunks['rec'].to_numpy()
n_per_rec = np.bincount(chunk_rec, minlength=len(rec))[:, None]

def recording_mean(chunk_probs):
    out = np.zeros((len(rec), chunk_probs.shape[1]), dtype=np.float64)
    np.add.at(out, chunk_rec, chunk_probs)
    return out / n_per_rec

text_emb = {}
for name, fmt in PROMPTS.items():
    texts = [fmt(species.loc[c]) for c in classes]
    with torch.no_grad():
        t = as_embed(model.get_text_features(**tok(texts, padding=True, return_tensors='pt').to(DEVICE)), model.text_projection)
    t = torch.nn.functional.normalize(t, dim=-1).float().cpu().numpy()
    text_emb[name] = t
    logits = scale * emb @ t.T
    p = np.exp(logits - logits.max(axis=1, keepdims=True)); p /= p.sum(axis=1, keepdims=True)
    np.save(OUT / f'probs_m3_{name}.npy', recording_mean(p).astype(np.float32))
    print(name, texts[0])
np.save(OUT / 'text_emb.npy', np.stack([text_emb[k] for k in PROMPTS]))

## 3. M4 — linear probe (out-of-fold)

Recording embedding = mean of its chunk embeddings, re-normalised. Standardise → multinomial logistic regression, `C = 1.0` fixed (not tuned), no class weights. Trained on folds ≠ k, predicts fold k.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = np.zeros((len(rec), emb.shape[1])); np.add.at(X, chunk_rec, emb)
X /= np.linalg.norm(X, axis=1, keepdims=True)
y = rec['label_idx'].to_numpy()

probs_m4 = np.zeros((len(rec), len(classes)), dtype=np.float32)
for k in sorted(rec['fold'].unique()):
    tr, te = rec['fold'] != k, rec['fold'] == k
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))
    clf.fit(X[tr], y[tr])
    missing = len(classes) - len(clf.classes_)
    if missing:
        print(f'fold {k}: {missing} classes absent from training (prob 0)')
    P = np.zeros((int(te.sum()), len(classes)), dtype=np.float32)
    P[:, clf.classes_] = clf.predict_proba(X[te])
    probs_m4[te.to_numpy()] = P
    print('fold', k, 'done')
np.save(OUT / 'probs_m4.npy', probs_m4)

## 4. Save and sanity-check

Only aggregate numbers here; per-species analysis is in `05_analysis` (pre-registered).

In [ ]:
out_rec = rec[['filename', 'primary_label', 'label_idx', 'fold', 'group', 'collection', 'n_secondary', 'duration', 'leak']]
out_rec.to_csv(OUT / 'recordings.csv', index=False)
(OUT / 'classes.json').write_text(json.dumps(classes))

from sklearn.metrics import f1_score, top_k_accuracy_score
labels = np.arange(len(classes))
for name in ['m3_P1', 'm3_P2', 'm3_P3', 'm4']:
    P = np.load(OUT / f'probs_{name}.npy')
    pred = P.argmax(1)
    print(f'{name:6s} acc={np.mean(pred == y):.3f}  macroF1={f1_score(y, pred, average="macro", labels=labels):.3f}  '
          f'top5={top_k_accuracy_score(y, P, k=5, labels=labels):.3f}')
print('chance top-1:', round(1 / len(classes), 4))